# Week 2: Creating Financial Forecasts\nTarget: daily closing price. Compare a 20-day moving average with linear regression using a chronological 80/20 split.

In [ ]:
from pathlib import Path\nimport pandas as pd, numpy as np, matplotlib.pyplot as plt\nROOT=Path.cwd()\nif not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent\nfiles=list((ROOT/'data').glob('*.xlsx'))\nraw=pd.read_excel(files[0]) if files else pd.read_csv(ROOT/'data/cleaned_financial_dataset.csv')\nraw.columns=[str(c).strip() for c in raw.columns]\nren={}\nfor c in raw.columns:\n k=c.lower().replace(' ','')\n if k=='date': ren[c]='Date'\n elif k.startswith('close'): ren[c]='Close'\ndf=raw.rename(columns=ren); df['Date']=pd.to_datetime(df['Date'],errors='coerce'); df['Close']=pd.to_numeric(df['Close'],errors='coerce')\ndf=df.dropna(subset=['Date','Close']).sort_values('Date').drop_duplicates('Date').reset_index(drop=True)\nprint(len(df),df.Date.min(),df.Date.max()); df.head()

## Historical trend

In [ ]:
plt.figure(figsize=(10,5)); plt.plot(df.Date,df.Close); plt.title('Historical Closing Price'); plt.xlabel('Date'); plt.ylabel('Close'); plt.tight_layout(); plt.show()

## Train/test and metrics

In [ ]:
n=len(df); split=int(n*.8); tr=df.iloc[:split]; te=df.iloc[split:]; close=df.Close.to_numpy(float)\nma=np.array([close[max(0,i-20):i].mean() for i in range(split,n)])\nslope,intercept=np.polyfit(np.arange(split),tr.Close.to_numpy(float),1); lr=intercept+slope*np.arange(split,n); y=te.Close.to_numpy(float)\ndef metrics(y,p):\n e=y-p; return {'MAE':np.mean(abs(e)),'RMSE':np.sqrt(np.mean(e**2)),'MAPE (%)':np.mean(abs(e/y))*100}\nscores=pd.DataFrame([{'Model':'20-day Moving Average',**metrics(y,ma)},{'Model':'Linear Regression',**metrics(y,lr)}]); scores

In [ ]:
plt.figure(figsize=(10,5)); plt.plot(te.Date,y,label='Actual'); plt.plot(te.Date,ma,label='Moving average'); plt.plot(te.Date,lr,label='Linear regression'); plt.legend(); plt.title('Actual vs Predicted — Test Period'); plt.tight_layout(); plt.show()

## Next 30 business-day forecast\nForecast dates are approximate business days and exclude exchange-specific holidays. These forecasts are educational estimates.

In [ ]:
hist=list(close[-20:]); maf=[]\nfor _ in range(30):\n v=float(np.mean(hist[-20:])); maf.append(v); hist.append(v)\nlrf=intercept+slope*np.arange(n,n+30); dates=pd.bdate_range(df.Date.iloc[-1]+pd.offsets.BDay(1),periods=30)\nforecast=pd.DataFrame({'Forecast_Date':dates,'Moving_Average_Forecast':maf,'Linear_Regression_Forecast':lrf}); forecast['Model_Average_Forecast']=forecast.iloc[:,1:3].mean(axis=1); forecast

In [ ]:
plt.figure(figsize=(10,5)); plt.plot(df.Date.tail(180),df.Close.tail(180),label='Recent history'); plt.plot(forecast.Forecast_Date,forecast.Moving_Average_Forecast,'--',label='Moving average'); plt.plot(forecast.Forecast_Date,forecast.Linear_Regression_Forecast,'--',label='Linear regression'); plt.legend(); plt.title('Next 30 Business-Day Forecast'); plt.tight_layout(); plt.show()

## Interpretation\nCompare MAE, RMSE and MAPE on the held-out test period. The lower error model performed better only on this test window; it is not guaranteed to be better in future periods. Forecasts are not investment advice.